# Selective next-FX-renko-bar direction research

This notebook consumes the reusable `range_bars_ml` library; FX feature construction and training logic do not live here.


In [1]:
# Reload local modelling modules so re-running this cell picks up source edits.
import importlib
import range_bars_ml as _range_bars_package
import range_bars_ml.model as _range_bars_model
import range_bars_ml.pipeline as _range_bars_pipeline
import range_bars_ml.fx_features as _range_bars_fx_features
import range_bars_ml.fx_pipeline as _range_bars_fx_pipeline

importlib.reload(_range_bars_model)
importlib.reload(_range_bars_pipeline)
importlib.reload(_range_bars_fx_features)
importlib.reload(_range_bars_fx_pipeline)
importlib.reload(_range_bars_package)

from range_bars_ml import (
    ExperimentConfig,
    FxFeatureConfig,
    ModelConfig,
    load_fx_renko_bars,
    prepare_fx_dataset,
    run_fx_experiment,
)
from range_bars_ml.model import signal_metrics
import numpy as np
import polars as pl


In [2]:
# Data selection
DATA_PATH = r"D:\Trading\2026\sp500_range_0.001.parquet"  # Set this to your FX parquet file.
train_end = pl.datetime(2024, 1, 1, time_zone='UTC')
test_end = pl.datetime(2025,1,1, time_zone='UTC')

# Causal feature definition
FEATURE_CONFIG = FxFeatureConfig(
    windows=range(1,50,1),
    time_column="close_timestamp",
)

# LightGBM and probability-calibration definition
MODEL_CONFIG = ModelConfig(
    calibration_fraction=0.10,
    random_state=416,
    n_estimators=200,
    learning_rate=0.03,
    num_leaves=100,
)

# Require a probability edge above the development-period up rate.
# Set to 0.0 to disable this additional anti-trend gate.
EDGE_MARGIN = 0.03

# Chronological evaluation and selective-signal definition
EXPERIMENT_CONFIG = ExperimentConfig(
    feature=FEATURE_CONFIG,
    model=MODEL_CONFIG,
    test_fraction=0.10,
    n_folds=5,
    # Zero permits an all-abstain strategy if no calls have positive net score.
    minimum_coverage=0.0,
    abstention_penalty=-0.0000001,
    edge_margin=EDGE_MARGIN,
)

# Inspection and artifact choices
FEATURE_PREVIEW_ROWS = 10
FEATURE_EXPORT_PATH = None  # e.g. r"C:\Code\Trading\2026\ml\data\fx_renko_features.parquet"
FEATURE_EXPORT_COMPRESSION = "zstd"
MODEL_ARTIFACT_PATH = "fx_renko_model.joblib"


In [3]:
all_bars = load_fx_renko_bars(DATA_PATH)
all_bars = all_bars.rename({'close_time':'close_timestamp'})


In [4]:
bars = all_bars.filter(pl.col("close_timestamp") < train_end)
#bars = bars.slice(offset=DATA_SLICE_OFFSET, length=DATA_SLICE_LENGTH)

In [5]:
bars.shape

(466909, 15)

In [6]:
bars[0]

timestamp,close_timestamp,tick_count,mid_open,mid_high,mid_low,mid_close,bid_open,bid_high,bid_low,bid_close,ask_open,ask_high,ask_low,ask_close
"datetime[μs, UTC]","datetime[μs, UTC]",i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2012-01-16 00:00:07.771 UTC,2012-01-16 00:02:05.820 UTC,17,1291.15,1291.35,1289.35,1289.35,1290.9,1291.1,1289.1,1289.1,1291.4,1291.6,1289.6,1289.6


In [7]:
# Inspect the fully causal, next-bar-labelled training frame.
feature_df, feature_names = prepare_fx_dataset(bars, config=FEATURE_CONFIG)
feature_df.head(FEATURE_PREVIEW_ROWS)

# Set FEATURE_EXPORT_PATH above to persist the complete feature frame.
if FEATURE_EXPORT_PATH:
    feature_df.write_parquet(FEATURE_EXPORT_PATH, compression=FEATURE_EXPORT_COMPRESSION)


In [8]:
feature_df

tick_count,bid_open,bid_high,bid_low,bid_close,ask_open,ask_high,ask_low,ask_close,bar_return,bar_range_pct,close_in_range,upper_wick_share,lower_wick_share,body_to_range,bar_direction,log_bar_duration,close_hour_sin,close_hour_cos,close_weekday_sin,close_weekday_cos,log_return_1,return_mean_1,return_std_1,direction_persistence_1,reversal_rate_1,direction_streak_share_1,trend_strength_1,return_acceleration_1,close_mean_zscore_1,channel_position_1,range_mean_1,range_std_1,range_regime_1,duration_log_mean_1,duration_log_std_1,duration_log_zscore_1,…,ema_fast_slow_gap_31_32,volatility_ratio_31_32,ema_fast_slow_gap_32_33,volatility_ratio_32_33,ema_fast_slow_gap_33_34,volatility_ratio_33_34,ema_fast_slow_gap_34_35,volatility_ratio_34_35,ema_fast_slow_gap_35_36,volatility_ratio_35_36,ema_fast_slow_gap_36_37,volatility_ratio_36_37,ema_fast_slow_gap_37_38,volatility_ratio_37_38,ema_fast_slow_gap_38_39,volatility_ratio_38_39,ema_fast_slow_gap_39_40,volatility_ratio_39_40,ema_fast_slow_gap_40_41,volatility_ratio_40_41,ema_fast_slow_gap_41_42,volatility_ratio_41_42,ema_fast_slow_gap_42_43,volatility_ratio_42_43,ema_fast_slow_gap_43_44,volatility_ratio_43_44,ema_fast_slow_gap_44_45,volatility_ratio_44_45,ema_fast_slow_gap_45_46,volatility_ratio_45_46,ema_fast_slow_gap_46_47,volatility_ratio_46_47,ema_fast_slow_gap_47_48,volatility_ratio_47_48,ema_fast_slow_gap_48_49,volatility_ratio_48_49,target
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8
44,1299.6,1299.6,1298.1,1298.1,1300.1,1300.1,1298.6,1298.6,-0.001154,0.001154,0.0,0.0,0.0,1.0,-1,12.868629,1.2246e-16,-1.0,0.433884,-0.900969,-0.001001,-0.001001,0.0,-1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.001154,0.0,0.0,12.868629,0.0,0.0,…,0.000058,1.012552,0.000058,0.998164,0.000058,0.99695,0.000057,0.99054,0.000057,1.003676,0.000056,0.993659,0.000056,1.012389,0.000055,1.006337,0.000055,1.006413,0.000054,1.011255,0.000054,1.001756,0.000053,1.002387,0.000052,1.003198,0.000052,1.001883,0.000051,1.001694,0.000051,0.996908,0.00005,1.001299,0.000049,0.997175,0
20,1297.9,1298.4,1297.1,1297.1,1298.4,1298.9,1297.6,1297.6,-0.000616,0.001001,0.0,0.384615,0.0,0.615385,-1,12.368085,1.2246e-16,-1.0,0.433884,-0.900969,-0.000771,-0.000771,0.0,-1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.001001,0.0,0.0,12.368085,0.0,0.0,…,0.000054,1.001541,0.000054,1.012499,0.000054,0.998825,0.000054,0.996091,0.000054,0.989736,0.000053,1.004048,0.000053,0.994363,0.000053,1.012204,0.000052,1.00573,0.000052,1.005834,0.000051,1.010813,0.000051,1.001197,0.00005,1.002702,0.00005,1.002695,0.000049,1.001395,0.000049,1.00201,0.000048,0.997357,0.000048,1.000871,0
37,1295.4,1296.4,1295.1,1296.4,1295.9,1296.9,1295.6,1296.9,0.000772,0.001003,1.0,0.0,0.230769,0.769231,1,12.594327,1.2246e-16,-1.0,0.433884,-0.900969,0.000617,0.000617,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.001003,0.0,0.0,12.594327,0.0,0.0,…,0.000044,1.005594,0.000044,0.99469,0.000045,1.0008,0.000045,1.011989,0.000045,0.999285,0.000046,0.995669,0.000046,0.98951,0.000046,1.004167,0.000046,0.995001,0.000046,1.011684,0.000045,1.005147,0.000045,1.005263,0.000045,1.010186,0.000045,1.000773,0.000045,1.002866,0.000044,1.002269,0.000044,1.001007,0.000044,1.00221,0
40,1296.1,1296.9,1295.6,1295.6,1296.6,1297.4,1296.1,1296.1,-0.000386,0.001003,0.0,0.615385,0.0,0.384615,-1,13.499467,1.2246e-16,-1.0,0.433884,-0.900969,-0.000617,-0.000617,0.0,-1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.001003,0.0,0.0,13.499467,0.0,0.0,…,0.000039,0.99617,0.00004,1.004751,0.00004,0.993815,0.000041,0.999995,0.000041,1.011826,0.000042,0.999556,0.000042,0.994968,0.000042,0.988829,0.000042,1.004294,0.000042,0.995334,0.000042,1.011478,0.000042,1.004652,0.000042,1.004789,0.000042,1.009816,0.000042,1.000305,0.000042,1.002994,0.000042,1.001847,0.000042,1.000594,1
27,1295.9,1297.4,1295.9,1297.4,1296.4,1297.9,1296.4,129

In [9]:
# Metrics include score (+1 correct, -1 wrong, -0.001 abstain).
result = run_fx_experiment(bars, config=EXPERIMENT_CONFIG)

# The edge gate is applied inside run_fx_experiment before its single final-test score.
dataset, _ = prepare_fx_dataset(bars, config=FEATURE_CONFIG)
test = dataset[result.split_plan.test.tolist()]
development_up_rate = result.model.metadata["development_up_rate"]


In [10]:
# Compare against simple regime/trend baselines on the same final test.
test_target = test["target"].to_numpy()
baseline_metrics = {
    "always_up": signal_metrics(np.ones(len(test_target)), test_target, 0.5, -0.01, abstention_penalty=EXPERIMENT_CONFIG.abstention_penalty),
    "always_down": signal_metrics(np.zeros(len(test_target)), test_target, 1.01, 0.5, abstention_penalty=EXPERIMENT_CONFIG.abstention_penalty),
    "always_abstain": signal_metrics(np.full(len(test_target), 0.5), test_target, 1.01, -0.01, abstention_penalty=EXPERIMENT_CONFIG.abstention_penalty),
}
score_comparison = pl.DataFrame([
    {"strategy": "model_edge_gated", **result.test_metrics},
    *({"strategy": name, **metrics} for name, metrics in baseline_metrics.items()),
]).select(["strategy", "score", "mean_score", "signals", "correct", "wrong", "abstentions", "precision"])

print(f"Development up rate: {development_up_rate:.3%}; edge gates: long >= {result.model.long_threshold:.3f}, short <= {result.model.short_threshold:.3f}")
score_comparison

Development up rate: 50.019%; edge gates: long >= 0.541, short <= 0.469


strategy,score,mean_score,signals,correct,wrong,abstentions,precision
str,f64,f64,f64,f64,f64,f64,f64
"""model_edge_gated""",-0.003477,-9.9966e-8,12.0,6.0,6.0,34772.0,0.5
"""always_up""",366.0,0.010522,34784.0,17575.0,17209.0,0.0,0.505261
"""always_down""",-366.0,-0.010522,34784.0,17209.0,17575.0,0.0,0.494739
"""always_abstain""",-0.003478,-1.0000e-7,0.0,0.0,0.0,34784.0,NaN


In [11]:
result.model.save(MODEL_ARTIFACT_PATH)


In [12]:
from range_bars_ml.model import SelectiveSignalModel
model = SelectiveSignalModel.load(MODEL_ARTIFACT_PATH)
feature_config = FxFeatureConfig(**model.metadata["feature_config"])

In [13]:
new_bars = all_bars.filter((pl.col("close_timestamp") >= train_end) & (pl.col("close_timestamp") <= test_end))
dataset, _ = prepare_fx_dataset(new_bars, feature_config)

In [14]:
predictions = model.predict(dataset)
signals = dataset.select("target").hstack(predictions)

In [15]:
metrics = signal_metrics(
      signals["probability_up"].to_numpy(),
      signals["target"].to_numpy(),
      model.long_threshold,
      model.short_threshold,
      abstention_penalty=model.metadata.get("abstention_penalty", -0.001),
  )

In [16]:
print(metrics)

{'coverage': 0.00026562188724350887, 'signals': 4.0, 'correct': 1.0, 'wrong': 3.0, 'abstentions': 15055.0, 'score': -2.0015055, 'mean_score': -0.0001329109170595657, 'precision': 0.25, 'long_coverage': 0.00019921641543263165, 'long_precision': 0.3333333333333333, 'long_signals': 3.0, 'short_coverage': 6.640547181087722e-05, 'short_precision': 0.0, 'short_signals': 1.0}


In [35]:
feature_df.columns

['tick_count',
 'bid_open',
 'bid_high',
 'bid_low',
 'bid_close',
 'ask_open',
 'ask_high',
 'ask_low',
 'ask_close',
 'bar_return',
 'bar_range_pct',
 'close_in_range',
 'upper_wick_share',
 'lower_wick_share',
 'body_to_range',
 'bar_direction',
 'log_bar_duration',
 'close_hour_sin',
 'close_hour_cos',
 'close_weekday_sin',
 'close_weekday_cos',
 'log_return_1',
 'return_mean_1',
 'return_std_1',
 'direction_persistence_1',
 'reversal_rate_1',
 'direction_streak_share_1',
 'trend_strength_1',
 'return_acceleration_1',
 'close_mean_zscore_1',
 'channel_position_1',
 'range_mean_1',
 'range_std_1',
 'range_regime_1',
 'duration_log_mean_1',
 'duration_log_std_1',
 'duration_log_zscore_1',
 'drawdown_from_high_1',
 'rebound_from_low_1',
 'distance_to_prior_high_1',
 'distance_to_prior_low_1',
 'breakout_up_1',
 'breakout_down_1',
 'new_high_count_1',
 'new_low_count_1',
 'bars_since_rolling_high_1',
 'bars_since_rolling_low_1',
 'ema_distance_1',
 'ema_slope_1',
 'abs_return_mean_1',


In [36]:
bars[-1]


timestamp,close_timestamp,tick_count,mid_open,mid_high,mid_low,mid_close,bid_open,bid_high,bid_low,bid_close,ask_open,ask_high,ask_low,ask_close
"datetime[μs, UTC]","datetime[μs, UTC]",i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2023-12-29 20:59:58.754 UTC,2023-12-29 21:04:01.227 UTC,149,4772.0,4773.627,4768.3815,4768.3815,4771.752,4773.323,4768.082,4768.082,4772.248,4773.931,4768.681,4768.681


In [37]:
import pandas as pd
from range_bars_ml import SelectiveSignalModel

model = SelectiveSignalModel.load(MODEL_ARTIFACT_PATH)

importance = (
  pd.DataFrame({
      "feature": model.feature_columns,
      "split_count": model.estimator.feature_importances_,
      "gain": model.estimator.booster_.feature_importance(importance_type="gain"),
  })
  .sort_values("gain", ascending=False)
)

print(importance.head(30).to_string(index=False))

                         feature  split_count        gain
            volatility_ratio_4_5           98 1294.504900
          volatility_ratio_11_12           97 1272.324157
          volatility_ratio_15_16           93 1166.901007
          volatility_ratio_16_17           92 1163.831817
          volatility_ratio_10_11           85 1153.345806
           volatility_ratio_9_10           80 1095.401655
          volatility_ratio_14_15           80 1076.470124
            volatility_ratio_8_9           81 1074.634689
            volatility_ratio_5_6           82 1059.543010
          volatility_ratio_18_19           80 1051.896232
          volatility_ratio_17_18           79 1038.397278
              duration_log_std_3           79 1026.146987
          volatility_ratio_12_13           70  924.767197
            volatility_ratio_3_4           73  921.628891
                  range_regime_2           67  916.009648
            volatility_ratio_2_3           73  907.753212
            vo